# HeatScale-AU: ERA5 yearly preprocessing

This notebook converts the filtered hourly ERA5 catalogue (`ERA5.csv.gz`) into one NumPy file per **complete calendar year**.

### Output

Each yearly file is written to:

```text
/g/data/x77/ha2606/HeatScale-AU-Data/ERA5/YYYY.npy
```

Each `.npy` file is a structured NumPy array with:

- `datetime`: exact hourly `datetime64[ns]`
- `data`: `float32` array with shape `(9, latitude, longitude)` for each sample

The variable axis is fixed as:

```text
0 tas
1 huss
2 ps
3 uas
4 vas
5 rsds
6 rsus
7 rlds
8 rlus
```

ERA5 is kept on its **native 0.25° grid**, cropped to grid cells lying inside the BARRA-C2 geographic footprint. No interpolation or regridding is performed here.

Parallelisation is by **year**: one process handles one complete year and writes one final yearly file.

In [ ]:
# ============================================================
# 1. Imports and configuration
# ============================================================

import os

# Prevent numerical libraries from creating extra threads inside
# each multiprocessing worker.
for name in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ.setdefault(name, "1")

import gc
import importlib
import multiprocessing as mp
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm


# ---------------------------
# Input catalogue
# ---------------------------

ERA5_CSV = Path("ERA5.csv.gz")


# ---------------------------
# Output
# ---------------------------

DATA_ROOT = Path("/g/data/x77/ha2606/HeatScale-AU-Data")
ERA5_OUT = DATA_ROOT / "ERA5"
ERA5_OUT.mkdir(parents=True, exist_ok=True)


# ---------------------------
# Multiprocessing
# ---------------------------

N_WORKERS = 28
OVERWRITE = False


print("ERA5 catalogue :", ERA5_CSV)
print("Output directory:", ERA5_OUT)
print("Workers         :", N_WORKERS)
print("Overwrite       :", OVERWRITE)

In [ ]:
# ============================================================
# 2. Load the filtered ERA5 catalogue and define variables
# ============================================================

ERA5 = pd.read_csv(
    ERA5_CSV,
    compression="gzip",
    parse_dates=["datetime"],
)

ERA5 = (
    ERA5[["datetime", "variable", "path"]]
    .drop_duplicates()
    .sort_values(["datetime", "variable"])
    .reset_index(drop=True)
)


# Native ERA5 inputs used to construct the study variables.
ERA5_SOURCE_VARIABLES = [
    "2t",
    "2d",
    "sp",
    "10u",
    "10v",
    "msdwswrf",
    "msnswrf",
    "msdwlwrf",
    "msnlwrf",
]


# Final fixed variable order used in every yearly NumPy file.
TARGET_VARIABLES = [
    "tas",
    "huss",
    "ps",
    "uas",
    "vas",
    "rsds",
    "rsus",
    "rlds",
    "rlus",
]

UNITS = [
    "K",
    "kg kg-1",
    "Pa",
    "m s-1",
    "m s-1",
    "W m-2",
    "W m-2",
    "W m-2",
    "W m-2",
]


# Keep only the ERA5 fields needed for this processing stage.
ERA5 = ERA5[
    ERA5["variable"].isin(ERA5_SOURCE_VARIABLES)
].copy()

ERA5["year"] = ERA5["datetime"].dt.year.astype(int)
ERA5["month"] = ERA5["datetime"].dt.month.astype(int)

YEARS = sorted(ERA5["year"].unique().tolist())

print(f"Catalogue rows : {len(ERA5):,}")
print(f"ERA5 period    : {YEARS[0]}-{YEARS[-1]}")
print(f"Catalogue years: {len(YEARS)}")
print("Target order   :", TARGET_VARIABLES)

In [ ]:
# ============================================================
# 3. Define the BARRA-C2 footprint and ERA5 cropped native grid
# ============================================================

BARRAC2_LTLN = {
    "lat_min": -45.69,
    "lat_max": -5.01,
    "lon_min": 108.02,
    "lon_max": 159.90,
    "dim": {"lat": 1018, "lon": 1298},
}

BBOX = {
    "lat_min": BARRAC2_LTLN["lat_min"],
    "lat_max": BARRAC2_LTLN["lat_max"],
    "lon_min": BARRAC2_LTLN["lon_min"],
    "lon_max": BARRAC2_LTLN["lon_max"],
}


# Read one ERA5 file only to obtain the native coordinate vectors.
first_era5_path = ERA5.loc[
    ERA5["variable"] == "2t",
    "path",
].iloc[0]

with xr.open_dataset(
    first_era5_path,
    decode_times=False,
    cache=False,
) as ds:
    latitude = np.asarray(ds["latitude"].values)
    longitude = np.asarray(ds["longitude"].values)


lat_index = np.where(
    (latitude >= BBOX["lat_min"])
    & (latitude <= BBOX["lat_max"])
)[0]

lon_index = np.where(
    (longitude >= BBOX["lon_min"])
    & (longitude <= BBOX["lon_max"])
)[0]

if len(lat_index) == 0 or len(lon_index) == 0:
    raise RuntimeError("No ERA5 grid cells fall inside the BARRA-C2 footprint.")

LAT_START = int(lat_index[0])
LAT_STOP = int(lat_index[-1] + 1)
LON_START = int(lon_index[0])
LON_STOP = int(lon_index[-1] + 1)

ERA5_LAT = latitude[LAT_START:LAT_STOP]
ERA5_LON = longitude[LON_START:LON_STOP]

ACTUAL_BBOX = np.array(
    [
        float(np.min(ERA5_LAT)),
        float(np.max(ERA5_LAT)),
        float(np.min(ERA5_LON)),
        float(np.max(ERA5_LON)),
    ],
    dtype=np.float32,
)

print("ERA5 cropped grid :", (len(ERA5_LAT), len(ERA5_LON)))
print("ERA5 latitude     :", float(ERA5_LAT.min()), "to", float(ERA5_LAT.max()))
print("ERA5 longitude    :", float(ERA5_LON.min()), "to", float(ERA5_LON.max()))

In [ ]:
# ============================================================
# 4. Save static ERA5 metadata once
# ============================================================

METADATA_FILE = ERA5_OUT / "metadata.npz"

np.savez(
    METADATA_FILE,
    variables=np.asarray(TARGET_VARIABLES),
    units=np.asarray(UNITS),
    latitude=np.asarray(ERA5_LAT, dtype=np.float32),
    longitude=np.asarray(ERA5_LON, dtype=np.float32),
    bbox=np.asarray(
        [
            BBOX["lat_min"],
            BBOX["lat_max"],
            BBOX["lon_min"],
            BBOX["lon_max"],
        ],
        dtype=np.float32,
    ),
    actual_bbox=ACTUAL_BBOX,
)

print("Saved:", METADATA_FILE)

## Year worker

The following cell writes a small importable Python module beside the notebook. This is intentional: with the `spawn` multiprocessing method, notebook-local worker functions are unreliable to pickle. The module lets each process independently handle one complete year.

In [ ]:
%%writefile era5_year_worker.py

import gc
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr


# ============================================================
# ERA5 NetCDF utilities
# ============================================================

def find_data_variable(ds):
    """Return the principal time-latitude-longitude data variable."""

    candidates = [
        name
        for name, da in ds.data_vars.items()
        if (
            "time" in da.dims
            and "latitude" in da.dims
            and "longitude" in da.dims
        )
    ]

    if len(candidates) != 1:
        raise ValueError(
            f"Expected one ERA5 data variable, found {candidates}"
        )

    return candidates[0]


# ============================================================
# ERA5 dew point -> specific humidity
# ============================================================

def huss_from_dewpoint(td, ps):
    """
    Convert ERA5 2 m dew-point temperature and surface pressure
    to specific humidity.

    Parameters
    ----------
    td : array
        2 m dew-point temperature [K]
    ps : array
        Surface pressure [Pa]

    Returns
    -------
    q : float32 array
        Specific humidity [kg kg-1]
    """

    td = np.asarray(td, dtype=np.float64)
    ps = np.asarray(ps, dtype=np.float64)

    Rd = 287.0597
    Rv = 461.5250
    epsilon = Rd / Rv

    a1 = 611.21
    a3 = 17.502
    a4 = 32.19
    T0 = 273.16

    # Vapour pressure from dew-point temperature.
    e = a1 * np.exp(
        a3 * (td - T0) / (td - a4)
    )

    # Specific humidity.
    q = (
        epsilon * e
        / (ps - (1.0 - epsilon) * e)
    )

    return q.astype(np.float32)


# ============================================================
# Read one source variable for one month
# ============================================================

def load_variable(paths, target_times, lat_slice, lon_slice):
    """Read and concatenate all matching data for one ERA5 source field."""

    target_times = pd.DatetimeIndex(target_times)

    data_parts = []
    time_parts = []

    for path in paths:
        with xr.open_dataset(
            path,
            decode_times=True,
            cache=False,
        ) as ds:
            varname = find_data_variable(ds)

            da = ds[varname].transpose(
                "time",
                "latitude",
                "longitude",
            )

            file_times = pd.DatetimeIndex(
                pd.to_datetime(ds["time"].values)
            )

            idx = np.where(
                file_times.isin(target_times)
            )[0]

            if len(idx) == 0:
                continue

            # Monthly ERA5 files are normally contiguous in time.
            if len(idx) == 1 or np.all(np.diff(idx) == 1):
                time_selector = slice(
                    int(idx[0]),
                    int(idx[-1] + 1),
                )
            else:
                time_selector = idx

            values = da.isel(
                time=time_selector,
                latitude=slice(*lat_slice),
                longitude=slice(*lon_slice),
            ).values

            data_parts.append(
                np.asarray(values, dtype=np.float32)
            )
            time_parts.append(file_times[idx].values)

    if not data_parts:
        raise RuntimeError("No matching ERA5 data were found.")

    data = np.concatenate(data_parts, axis=0)
    times = np.concatenate(time_parts)

    order = np.argsort(times)
    times = pd.DatetimeIndex(times[order])
    data = data[order]

    if not times.equals(target_times):
        missing = target_times.difference(times)
        extra = times.difference(target_times)
        raise RuntimeError(
            f"ERA5 datetime mismatch: missing={len(missing)}, extra={len(extra)}"
        )

    return data


# ============================================================
# Process one complete calendar year
# ============================================================

def process_year(task):
    year = int(task["year"])
    out_dir = Path(task["out_dir"])

    final_path = out_dir / f"{year}.npy"
    partial_path = out_dir / f"{year}.partial.npy"

    try:
        if final_path.exists() and not task["overwrite"]:
            return {
                "year": year,
                "status": "SKIPPED",
                "hours": len(task["times"]),
                "file": str(final_path),
                "error": None,
            }

        if partial_path.exists():
            partial_path.unlink()

        times = pd.DatetimeIndex(task["times"])

        ntime = len(times)
        nlat = int(task["nlat"])
        nlon = int(task["nlon"])

        lat_slice = tuple(task["lat_slice"])
        lon_slice = tuple(task["lon_slice"])

        # Each record contains its datetime and a (9, lat, lon) matrix.
        sample_dtype = np.dtype(
            [
                ("datetime", "datetime64[ns]"),
                ("data", np.float32, (9, nlat, nlon)),
            ]
        )

        year_data = np.empty(
            ntime,
            dtype=sample_dtype,
        )

        year_data["datetime"] = times.values.astype(
            "datetime64[ns]"
        )

        # ----------------------------------------------------
        # Fill the full year month by month.
        # ----------------------------------------------------
        for month in range(1, 13):
            month_mask = times.month == month
            positions = np.where(month_mask)[0]

            if len(positions) == 0:
                raise RuntimeError(
                    f"{year}-{month:02d}: no timestamps found"
                )

            start = int(positions[0])
            stop = int(positions[-1] + 1)
            month_times = times[month_mask]

            paths = task["month_paths"][month]

            # Native ERA5 source fields.
            t2 = load_variable(
                paths["2t"], month_times, lat_slice, lon_slice
            )
            td2 = load_variable(
                paths["2d"], month_times, lat_slice, lon_slice
            )
            sp = load_variable(
                paths["sp"], month_times, lat_slice, lon_slice
            )
            u10 = load_variable(
                paths["10u"], month_times, lat_slice, lon_slice
            )
            v10 = load_variable(
                paths["10v"], month_times, lat_slice, lon_slice
            )
            sw_down = load_variable(
                paths["msdwswrf"], month_times, lat_slice, lon_slice
            )
            sw_net = load_variable(
                paths["msnswrf"], month_times, lat_slice, lon_slice
            )
            lw_down = load_variable(
                paths["msdwlwrf"], month_times, lat_slice, lon_slice
            )
            lw_net = load_variable(
                paths["msnlwrf"], month_times, lat_slice, lon_slice
            )

            out = year_data["data"][start:stop]

            # Fixed target-variable order:
            # 0 tas, 1 huss, 2 ps, 3 uas, 4 vas,
            # 5 rsds, 6 rsus, 7 rlds, 8 rlus
            out[:, 0] = t2
            out[:, 1] = huss_from_dewpoint(td2, sp)
            out[:, 2] = sp
            out[:, 3] = u10
            out[:, 4] = v10
            out[:, 5] = sw_down

            # Surface net shortwave = downward - upward.
            out[:, 6] = sw_down - sw_net

            out[:, 7] = lw_down

            # Surface net longwave = downward - upward.
            out[:, 8] = lw_down - lw_net

            del (
                t2,
                td2,
                sp,
                u10,
                v10,
                sw_down,
                sw_net,
                lw_down,
                lw_net,
                out,
            )
            gc.collect()

        # Write a complete year to a temporary name first.
        np.save(
            partial_path,
            year_data,
            allow_pickle=False,
        )

        del year_data
        gc.collect()

        # Publish only after the complete yearly file has been written.
        os.replace(partial_path, final_path)

        return {
            "year": year,
            "status": "OK",
            "hours": ntime,
            "file": str(final_path),
            "error": None,
        }

    except Exception as exc:
        if partial_path.exists():
            try:
                partial_path.unlink()
            except Exception:
                pass

        return {
            "year": year,
            "status": "FAILED",
            "hours": len(task.get("times", [])),
            "file": None,
            "error": repr(exc),
        }


In [ ]:
# ============================================================
# 6. Import the yearly worker
# ============================================================

import era5_year_worker

importlib.reload(era5_year_worker)

from era5_year_worker import process_year

print("Loaded worker:", era5_year_worker.__file__)

In [ ]:
# ============================================================
# 7. Prepare one task for every COMPLETE calendar year
# ============================================================

# (year, month, variable) -> one or more source NetCDF paths
PATH_INDEX = (
    ERA5.groupby(["year", "month", "variable"])["path"]
    .unique()
    .to_dict()
)


def prepare_full_year_tasks(df):
    tasks = []
    skipped = []

    for year in sorted(df["year"].unique()):
        year = int(year)

        expected_times = pd.date_range(
            start=f"{year}-01-01 00:00:00",
            end=f"{year + 1}-01-01 00:00:00",
            freq="h",
            inclusive="left",
        )

        # A yearly output is created only if all nine ERA5 source
        # variables contain exactly the complete hourly calendar year.
        complete = True
        reason = None

        for variable in ERA5_SOURCE_VARIABLES:
            actual_times = pd.DatetimeIndex(
                df.loc[
                    (df["year"] == year)
                    & (df["variable"] == variable),
                    "datetime",
                ].unique()
            ).sort_values()

            if not actual_times.equals(expected_times):
                complete = False
                reason = (
                    f"{variable}: {len(actual_times):,} hours, "
                    f"expected {len(expected_times):,}"
                )
                break

        if not complete:
            skipped.append(
                {
                    "year": year,
                    "reason": reason,
                }
            )
            continue

        month_paths = {}

        for month in range(1, 13):
            month_paths[month] = {}

            for variable in ERA5_SOURCE_VARIABLES:
                key = (year, month, variable)

                if key not in PATH_INDEX:
                    raise RuntimeError(
                        f"Missing source path for {year}-{month:02d} {variable}"
                    )

                paths = [str(path) for path in PATH_INDEX[key]]

                if len(paths) == 0:
                    raise RuntimeError(
                        f"No source path for {year}-{month:02d} {variable}"
                    )

                month_paths[month][variable] = paths

        tasks.append(
            {
                "year": year,
                "times": expected_times.values,
                "month_paths": month_paths,
                "lat_slice": (LAT_START, LAT_STOP),
                "lon_slice": (LON_START, LON_STOP),
                "nlat": len(ERA5_LAT),
                "nlon": len(ERA5_LON),
                "out_dir": str(ERA5_OUT),
                "overwrite": OVERWRITE,
            }
        )

    return tasks, skipped


YEAR_TASKS, INCOMPLETE_YEARS = prepare_full_year_tasks(ERA5)

print(f"Complete years prepared: {len(YEAR_TASKS)}")

if INCOMPLETE_YEARS:
    print("Incomplete years excluded:")
    for item in INCOMPLETE_YEARS:
        print(" ", item["year"], "->", item["reason"])

In [ ]:
# ============================================================
# 8. Select years that actually need processing
# ============================================================

if OVERWRITE:
    RUN_TASKS = YEAR_TASKS
else:
    RUN_TASKS = [
        task
        for task in YEAR_TASKS
        if not (ERA5_OUT / f"{task['year']}.npy").exists()
    ]

EXISTING_YEARS = sorted(
    set(task["year"] for task in YEAR_TASKS)
    - set(task["year"] for task in RUN_TASKS)
)

print(f"Years ready       : {len(YEAR_TASKS)}")
print(f"Already completed : {len(EXISTING_YEARS)}")
print(f"Years to process  : {len(RUN_TASKS)}")

if RUN_TASKS:
    print(
        "Processing range  :",
        min(task["year"] for task in RUN_TASKS),
        "to",
        max(task["year"] for task in RUN_TASKS),
    )

In [ ]:
# ============================================================
# 9. Process ALL complete years in parallel
# ============================================================

RESULTS = []

if not RUN_TASKS:
    print("Nothing to process. All complete yearly files already exist.")

else:
    workers = min(N_WORKERS, len(RUN_TASKS))
    ctx = mp.get_context("spawn")

    print(f"Starting {workers} yearly worker processes...")

    with ctx.Pool(
        processes=workers,
        maxtasksperchild=1,
    ) as pool:

        iterator = pool.imap_unordered(
            process_year,
            RUN_TASKS,
            chunksize=1,
        )

        for result in tqdm(
            iterator,
            total=len(RUN_TASKS),
            desc="ERA5 complete years",
        ):
            RESULTS.append(result)

            if result["status"] == "FAILED":
                print(
                    f"\nFAILED {result['year']}: "
                    f"{result['error']}"
                )

In [ ]:
# ============================================================
# 10. Final processing summary
# ============================================================

if RESULTS:
    RESULTS_DF = (
        pd.DataFrame(RESULTS)
        .sort_values("year")
        .reset_index(drop=True)
    )

    display(RESULTS_DF)
    print("\nStatus counts:")
    print(RESULTS_DF["status"].value_counts())

    failed = RESULTS_DF[
        RESULTS_DF["status"] == "FAILED"
    ]

    if len(failed) > 0:
        print("\nFailed years:")
        display(failed[["year", "error"]])
    else:
        print("\nAll requested complete years finished successfully.")

else:
    RESULTS_DF = pd.DataFrame(
        columns=["year", "status", "hours", "file", "error"]
    )

    print("No new yearly processing was required.")

## Result layout

For a processed year such as `1979.npy`:

```python
ERA5_1979 = np.load(
    "/g/data/x77/ha2606/HeatScale-AU-Data/ERA5/1979.npy",
    mmap_mode="r",
)

ERA5_1979["datetime"].shape
ERA5_1979["data"].shape
```

The data field has layout:

```text
(time, variable, latitude, longitude)
```

The variable order and coordinates are stored once in `metadata.npz`.